CNN Training

In [ ]:
import os
from pathlib import Path
import pandas as pd
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from torchvision.models import ResNet18_Weights
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report



In [ ]:
if torch.cuda.is_available():
    device_name = "cuda"
elif torch.backends.mps.is_available():
    device_name = "mps"
else:
    device_name = "cpu"
    
print(f'Using device: {device_name}')
device = torch.device(device_name)

Load Dataset

In [ ]:
labels = pd.read_csv("../data/labels.csv")
image_paths = []
pattern_labels = []

for idx, row in labels.iterrows():
    folder = f"../data/frames/{row['division']}/{row['division']}_{row['id']}"
    for img_path in Path(folder).glob("*.jpg"):
        image_paths.append(str(img_path))
        pattern_labels.append(row['pattern'])


patterns = sorted(set(pattern_labels))
label2id = {p: i for i, p in enumerate(patterns)}

y = np.array([label2id[p] for p in pattern_labels])
len(image_paths)


Transform and Dataset class

In [ ]:
tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class FrameDataset(Dataset):
    def __init__(self, paths, labels, tfm):
        self.paths = paths
        self.labels = labels
        self.tfm = tfm

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.tfm(img), self.labels[idx]

Train/Test split and DataLoaders

In [ ]:
g = torch.Generator().manual_seed(42)
full_ds = FrameDataset(image_paths, y, tfm)
n = len(full_ds)
n_train = int(0.8 * n)
n_test = n - n_train


train_ds, test_ds = random_split(full_ds, [n_train, n_test], generator=g)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=32, shuffle=False)

n_train, n_test

CNN model

In [ ]:
num_classes = len(patterns)

model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


Training Loop

In [ ]:
def run_epoch(loader, train=True):
    model.train(train)
    total_loss = 0
    correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        if train:
            optimizer.zero_grad()

        logits = model(xb)
        loss = criterion(logits, yb)

        if train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * yb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        total += yb.size(0)

    return total_loss / total, correct / total


Train for specified number of epochs

In [ ]:
EPOCHS = 15

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

for epoch in range(1, EPOCHS+1):
    train_loss, train_acc = run_epoch(train_dl, train=True)
    val_loss, val_acc = run_epoch(test_dl, train=False)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:02d} | "
          f"Train {train_loss:.4f}, Acc {train_acc:.3f} | "
          f"Val {val_loss:.4f}, Acc {val_acc:.3f}")

In [ ]:
y_true = []
y_pred = []

model.eval()
with torch.no_grad():
    for xb, yb in test_dl:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        preds = logits.argmax(1)

        y_true.extend(yb.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print(classification_report(y_true, y_pred, target_names=patterns))

Save model

In [ ]:
Path("models").mkdir(exist_ok=True)

torch.save({
    "state_dict": model.state_dict(),
    "label2id": label2id,
    "id2label": {v: k for k, v in label2id.items()}
}, "models/cnn_sugarpush_sugartag.pth")

print("Saved model!")


In [ ]:
epochs = range(1, EPOCHS + 1)

plt.figure(figsize=(14, 5))

# Loss plot
plt.subplot(1, 2, 1)
plt.plot(epochs, history["train_loss"], label="Train Loss")
plt.plot(epochs, history["val_loss"], label="Val Loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

# Accuracy plot
plt.subplot(1, 2, 2)
plt.plot(epochs, history["train_acc"], label="Train Accuracy")
plt.plot(epochs, history["val_acc"], label="Val Accuracy")
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()